## STUDENT INFO:

#### Name:
#### 1. NUR FIRZANA BINTI BADRUS HISHAM (A23CS0156)
#### 2. NURUL ADRIANA BINTI KAMAL JEFRI (A23CS0258)
#### 3. HARINI A/P SANGAAN (A23CS0081)

#### Section: 02

## Documentation: Phase I - Data Preparation and Cleaning
#### Objective

#### 1.Prepare a clean and merged dataset using:
#####    - FuelConsumption.csv (Canada, 2014 CO2 emission data)
#####    - Cars2025.csv (Malaysia, 2025 car registrations)

#### The processed dataset will be used for emission-based analytics in Phase II.

## Part 1 : Dataset Matching & Initial Cleaning
#### Tasks Completed
#### 1. Data Loading:
#####     - Loaded FuelConsumption.csv and cars_2025.csv using pandas.read_csv.
#### 2. Standardization:
#####     - Converted maker and model columns in both datasets to uppercase and removed leading/trailing whitespace.
#### 3. Merging Strategy:
#####     - Performed a merge on standardized maker/MAKE and model/MODEL columns using inner join.
#### 4. Initial Cleaning:
#####     - Dropped rows with missing values in critical fields (e.g., CO2EMISSIONS, ENGINESIZE).
#####     - Exported cleaned data

In [4]:
# Step 1: Import Libraries

import pandas as pd
import numpy as np

In [6]:
# Step 2: Load Datasets

fuel_data = pd.read_csv('FuelConsumption.csv')  # 2014 emissions data
cars_data = pd.read_csv('cars_2025.csv')        # 2025 registration data

# Display first few rows to inspect
print("Fuel Consumption Data (2014):")
display(fuel_data.head(2))

print("\nCar Registrations Data (2025):")
display(cars_data.head(2))

Fuel Consumption Data (2014):


,MODELYEAR,MAKE,MODEL,VEHICLECLASS,ENGINESIZE,CYLINDERS,TRANSMISSION,FUELTYPE,FUELCONSUMPTION_CITY,FUELCONSUMPTION_HWY,FUELCONSUMPTION_COMB,FUELCONSUMPTION_COMB_MPG,CO2EMISSIONS
0,2014,ACURA,ILX,COMPACT,2.0,4,AS5,Z,9.9,6.7,8.5,33,196
1,2014,ACURA,ILX,COMPACT,2.4,4,M6,Z,11.2,7.7,9.6,29,221



Car Registrations Data (2025):


,date_reg,type,maker,model,colour,fuel,state
0,2025-01-01,motokar,BYD,Seal,white,electric,Rakan Niaga
1,2025-01-01,window_van,Cam,Placer-X,yellow,greendiesel,Johor


In [20]:
# Clean 'maker' and 'model' columns in df_cars
cars_data['maker'] = cars_data['maker'].str.upper().str.strip()
cars_data['model'] = cars_data['model'].str.upper().str.strip()

# Clean 'MAKE' and 'MODEL' columns in df_fuel and rename them
fuel_data_cleaned = fuel_data.copy()
fuel_data_cleaned['MAKE'] = fuel_data_cleaned['MAKE'].str.upper().str.strip()
fuel_data_cleaned['MODEL'] = fuel_data_cleaned['MODEL'].str.upper().str.strip()
fuel_data_cleaned = fuel_data_cleaned.rename(columns={'MAKE': 'maker', 'MODEL': 'model'})

# Attempt to merge again after cleaning
merged_data = pd.merge(
    cars_data, 
    fuel_data_cleaned, 
    on=['maker', 'model'], 
    how='inner')

# Display the first few rows of the merged dataframe
print("\nMerged DataFrame head after cleaning:")
display(merged_data.head())

# Display the shape of the merged dataframe
print("\nShape of Merged DataFrame after cleaning:", merged_data.shape)


Merged DataFrame head after cleaning:


,date_reg,type,maker,model,colour,fuel,state,MODELYEAR,VEHICLECLASS,ENGINESIZE,CYLINDERS,TRANSMISSION,FUELTYPE,FUELCONSUMPTION_CITY,FUELCONSUMPTION_HWY,FUELCONSUMPTION_COMB,FUELCONSUMPTION_COMB_MPG,CO2EMISSIONS
0,2025-01-01,motokar,PORSCHE,911 CARRERA,grey,petrol,Johor,2014,MINICOMPACT,3.4,6,AM7,Z,11.6,8.3,10.1,28,232
1,2025-01-01,motokar,PORSCHE,911 CARRERA,grey,petrol,Johor,2014,MINICOMPACT,3.4,6,M7,Z,12.5,8.6,10.7,26,246
2,2025-01-02,motokar,HONDA,CIVIC,white,petrol,Selangor,2014,COMPACT,1.8,4,AV,X,7.9,6.0,7.0,40,161
3,2025-01-02,motokar,HONDA,CIVIC,white,petrol,Selangor,2014,COMPACT,1.8,4,AV7,X,8.1,6.2,7.2,39,166
4,2025-01-02,motokar,HONDA,CIVIC,white,petrol,Selangor,2014,COMPACT,1.8,4,M5,X,8.5,6.6,7.6,37,175



Shape of Merged DataFrame after cleaning: (25863, 18)


In [28]:
# Step 5: Export Cleaned Data
merged_data.to_csv('Cleaned_Merged_Cars_FIXED.csv', index=False)

## Part 2: Feature Engineering & Aggregated Analysis
#### 1. Emission Category Feature: CO2_Emission_Category
#####     - To categorize vehicles into "Low", "Medium", and "High" CO2 emission groups based on CO2EMISSIONS distribution.
#### 2. Aggregated Summaries by Key Attributes
##### Each aggregation includes:
#####   - Average CO2 Emissions
#####   - Average Combined Fuel Consumption
#####   - Total Registrations (count of rows)
#####   - Aggregate by VEHICLECLASS, ENGINESIZE, CYLINDERS, TRANSMISSION, FUELTYPE, TOP 10 CARS BY AVERAGE CO2 EMISSION AND CO2 EMISSION CATEGORY
#### 3. Final Export
#####   - The complete, cleaned, and engineered dataset is saved as Cleaned_Aggregated_Merged_Cars_Final.csv

In [32]:
# --- Emission_Category: "Low", "Medium", "High" based on CO2 thresholds ---
# This was already implemented and is re-verified here.
print("\nCategorizing CO2 Emissions into 'Low', 'Medium', 'High':")
if 'CO2EMISSIONS' in merged_data.columns and not merged_data['CO2EMISSIONS'].empty:
    bins = [merged_data['CO2EMISSIONS'].min() - 1,
            merged_data['CO2EMISSIONS'].quantile(0.33),
            merged_data['CO2EMISSIONS'].quantile(0.66),
            merged_data['CO2EMISSIONS'].max() + 1]
    labels = ['Low Emission', 'Medium Emission', 'High Emission']
    merged_data['CO2_Emission_Category'] = pd.cut(merged_data['CO2EMISSIONS'], bins=bins, labels=labels, right=True)

    print("Distribution of CO2 Emission Categories:")
    display(merged_data['CO2_Emission_Category'].value_counts())
else:
    print("CO2EMISSIONS column is empty or missing, skipping categorization.")


Categorizing CO2 Emissions into 'Low', 'Medium', 'High':
Distribution of CO2 Emission Categories:


CO2_Emission_Category
Low Emission       10153
High Emission       8358
Medium Emission     7352
Name: count, dtype: int64

In [48]:
# Step 8: Data Aggregation and Group Operations (and Exporting Tables)

# Aggregation 1: Average CO2 Emissions and Fuel Consumption by Vehicle Class
print("\nSummary: Average CO2 Emissions and Fuel Consumption by Vehicle Class:")
vehicle_class_summary = merged_data.groupby('VEHICLECLASS').agg(
    Avg_CO2_Emissions=('CO2EMISSIONS', 'mean'),
    Avg_Fuel_Consumption=('FUELCONSUMPTION_COMB', 'mean'),
    Total_Registrations=('maker', 'count') # Changed from 'registration_id' to 'maker' for robustness
).sort_values(by='Avg_CO2_Emissions', ascending=False)
display(vehicle_class_summary)
vehicle_class_summary.to_csv('Average_Emission_by_VehicleClass.csv', index=True)
print("Exported 'Average_Emission_by_VehicleClass.csv'")


Summary: Average CO2 Emissions and Fuel Consumption by Vehicle Class:


,Avg_CO2_Emissions,Avg_Fuel_Consumption,Total_Registrations
VEHICLECLASS,,,
SUV - STANDARD,294.342239,12.792875,786
SUBCOMPACT,282.318182,12.268182,88
FULL-SIZE,251.000000,10.900000,21
MINIVAN,244.177778,10.607778,90
TWO-SEATER,233.769231,10.151603,312
MINICOMPACT,232.128631,10.092946,241
SUV - SMALL,208.455960,9.064119,7391
MID-SIZE,199.044170,8.645230,566
STATION WAGON - SMALL,174.500000,7.600000,22


Exported 'Average_Emission_by_VehicleClass.csv'


In [66]:
# Aggregation 2: Average CO2 Emissions and Fuel Consumption by Engine Size
print("\nSummary: Average CO2 Emissions and Fuel Consumption by Engine Size:")
engine_size_summary = merged_data.groupby('ENGINESIZE').agg(
    Avg_CO2_Emissions=('CO2EMISSIONS', 'mean'),
    Avg_Fuel_Consumption=('FUELCONSUMPTION_COMB', 'mean'),
    Total_Registrations=('maker', 'count') # Changed from 'registration_id' to 'maker' for robustness
).sort_values(by='Avg_CO2_Emissions', ascending=False)
display(engine_size_summary)
engine_size_summary.to_csv('Average_Emission_by_EngineSize.csv', index=True)
print("Exported 'Average_Emission_by_EngineSize.csv'")


Summary: Average CO2 Emissions and Fuel Consumption by Engine Size:


,Avg_CO2_Emissions,Avg_Fuel_Consumption,Total_Registrations
ENGINESIZE,,,
6.8,437.000000,19.000000,1
5.2,381.500000,16.600000,2
4.2,371.500000,16.150000,2
5.9,359.000000,15.600000,1
6.0,356.000000,15.500000,2
4.7,347.000000,15.100000,1
5.8,304.000000,13.200000,10
5.7,301.000000,13.100000,2
3.8,297.421053,12.921053,57


Exported 'Average_Emission_by_EngineSize.csv'


In [64]:
# Aggregation 3: Average CO2 Emissions and Fuel Consumption by Cylinders
print("\nSummary: Average CO2 Emissions and Fuel Consumption by Cylinders:")
cylinders_summary = merged_data.groupby('CYLINDERS').agg(
    Avg_CO2_Emissions=('CO2EMISSIONS', 'mean'),
    Avg_Fuel_Consumption=('FUELCONSUMPTION_COMB', 'mean'),
    Total_Registrations=('maker', 'count') # Changed for robustness
).sort_values(by='Avg_CO2_Emissions', ascending=False)
display(cylinders_summary)
cylinders_summary.to_csv('Average_Emission_by_Cylinders.csv', index=True)
print("Exported 'Average_Emission_by_Cylinders.csv'")


Summary: Average CO2 Emissions and Fuel Consumption by Cylinders:


,Avg_CO2_Emissions,Avg_Fuel_Consumption,Total_Registrations
CYLINDERS,,,
10,381.500000,16.600000,2
12,357.000000,15.533333,3
8,294.636364,12.811364,44
6,269.228361,11.698956,1629
5,223.000000,9.700000,6
4,179.605939,7.805654,24179


Exported 'Average_Emission_by_Cylinders.csv'


In [62]:
# Aggregation 4: Average CO2 Emissions and Fuel Consumption by Transmission Type
print("\nSummary: Average CO2 Emissions and Fuel Consumption by Transmission Type:")
transmission_summary = merged_data.groupby('TRANSMISSION').agg(
    Avg_CO2_Emissions=('CO2EMISSIONS', 'mean'),
    Avg_Fuel_Consumption=('FUELCONSUMPTION_COMB', 'mean'),
    Total_Registrations=('maker', 'count') # Changed for robustness
).sort_values(by='Avg_CO2_Emissions', ascending=False)
display(transmission_summary)
transmission_summary.to_csv('Average_Emission_by_TransmissionType.csv', index=True)
print("Exported 'Average_Emission_by_TransmissionType.csv'")


Summary: Average CO2 Emissions and Fuel Consumption by Transmission Type:


,Avg_CO2_Emissions,Avg_Fuel_Consumption,Total_Registrations
TRANSMISSION,,,
A7,345.000000,15.000000,2
AM6,297.000000,12.900000,33
AS8,276.977778,12.033333,45
A8,276.000000,12.000000,385
AM7,248.121951,10.792683,123
M7,246.151899,10.706329,79
A6,244.173975,10.614286,707
A9,230.000000,10.000000,2
M6,229.611218,9.979243,2086


Exported 'Average_Emission_by_TransmissionType.csv'


In [70]:
# Aggregation 5: Average CO2 Emissions and Fuel Consumption by Fuel Type
print("\nSummary: Average CO2 Emissions and Fuel Consumption by Fuel Type:")
fuel_type_summary = merged_data.groupby('FUELTYPE').agg(
    Avg_CO2_Emissions=('CO2EMISSIONS', 'mean'),
    Avg_Fuel_Consumption=('FUELCONSUMPTION_COMB', 'mean'),
    Total_Registrations=('maker', 'count') # Changed for robustness
).sort_values(by='Avg_CO2_Emissions', ascending=False)
display(fuel_type_summary)
fuel_type_summary.to_csv('Average_Emission_by_FuelType.csv', index=True)
print("Exported 'Average_Emission_by_FuelType.csv'")


Summary: Average CO2 Emissions and Fuel Consumption by Fuel Type:


,Avg_CO2_Emissions,Avg_Fuel_Consumption,Total_Registrations
FUELTYPE,,,
Z,265.298372,11.530172,2088
X,178.484038,7.756770,23775


Exported 'Average_Emission_by_FuelType.csv'


In [74]:
# Aggregation 6: Top 10 Car Makers by Average CO2 Emissions
print("\nSummary: Top 10 Car Makers by Average CO2 Emissions:")
maker_emissions = merged_data.groupby('maker').agg(
    Avg_CO2_Emissions=('CO2EMISSIONS', 'mean'),
    Total_Registrations=('maker', 'count') # Changed for robustness
).sort_values(by='Avg_CO2_Emissions', ascending=False).head(10)
display(maker_emissions)
maker_emissions.to_csv('Average_Emission_by_Make.csv', index=True)
print("Exported 'Average_Emission_by_Make.csv'")


Summary: Top 10 Car Makers by Average CO2 Emissions:


,Avg_CO2_Emissions,Total_Registrations
maker,,
ASTON MARTIN,359.000000,1
BENTLEY,348.600000,5
MASERATI,347.000000,1
NISSAN,297.000000,33
PORSCHE,284.729145,971
DODGE,279.500000,4
AUDI,273.242424,33
FORD,266.568627,51
VOLVO,257.520548,146


Exported 'Average_Emission_by_Make.csv'


In [78]:
# Aggregation 7: Top 10 Car Models by Average CO2 Emissions
print("\nSummary: Top 10 Car Models (Maker + Model) by Average CO2 Emissions:")
model_emissions = merged_data.groupby(['maker', 'model']).agg(
    Avg_CO2_Emissions=('CO2EMISSIONS', 'mean'),
    Total_Registrations=('maker', 'count') # Changed for robustness
).sort_values(by='Avg_CO2_Emissions', ascending=False).head(10)
display(model_emissions)
model_emissions.to_csv('Average_Emission_by_Model.csv', index=True)
print("Exported 'Average_Emission_by_Model.csv'")


Summary: Top 10 Car Models (Maker + Model) by Average CO2 Emissions:


,,Avg_CO2_Emissions,Total_Registrations
maker,model,,
BENTLEY,MULSANNE,437.0,1
AUDI,R8,376.5,4
ASTON MARTIN,VANQUISH,359.0,1
MASERATI,GRANTURISMO,347.0,1
BENTLEY,CONTINENTAL GT,326.5,4
PORSCHE,911 GT3,322.0,12
AUDI,Q7,304.0,4
NISSAN,GT-R,297.0,33
AUDI,S8,297.0,1


Exported 'Average_Emission_by_Model.csv'


In [84]:
# Aggregation 8: Group by CO2 Emission Category
if 'CO2_Emission_Category' in merged_data.columns:
    print("\nSummary: Average Engine Size and Cylinders by CO2 Emission Category:")
    emission_category_summary = merged_data.groupby('CO2_Emission_Category').agg(
        Avg_Engine_Size=('ENGINESIZE', 'mean'),
        Avg_Cylinders=('CYLINDERS', 'mean'),
        Total_Cars=('maker', 'count') # Changed for robustness
    )
    display(emission_category_summary)
    emission_category_summary.to_csv('Summary_by_CO2_Emission_Category.csv', index=True)
    print("Exported 'Summary_by_CO2_Emission_Category.csv'")
else:
    print("Skipping 'Summary_by_CO2_Emission_Category' as 'CO2_Emission_Category' column is not available.")


Summary: Average Engine Size and Cylinders by CO2 Emission Category:


C:\Users\harin\AppData\Local\Temp\ipykernel_17224\3524412058.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  emission_category_summary = merged_data.groupby('CO2_Emission_Category').agg(


,Avg_Engine_Size,Avg_Cylinders,Total_Cars
CO2_Emission_Category,,,
Low Emission,1.732956,4.000000,10153
Medium Emission,1.734752,4.000000,7352
High Emission,2.546231,4.415889,8358


Exported 'Summary_by_CO2_Emission_Category.csv'


In [86]:
# Final export of the complete cleaned and engineered data for Phase II.
final_cleaned_data_path = 'Cleaned_Aggregated_Merged_Cars_Final.csv'
merged_data.to_csv(final_cleaned_data_path, index=False)
print(f"\nFinal cleaned and aggregated data saved to: {final_cleaned_data_path}")


Final cleaned and aggregated data saved to: Cleaned_Aggregated_Merged_Cars_Final.csv
